# L16 · 실패 진단·평가·capstone

## Goal

- reward hacking과 collapse를 지표로 찾는다
- 공정 비교 조건을 감사한다
- local 결과와 논문 수치를 분리한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L16:toy:42").hexdigest()
print(f"lesson=L16 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L16 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:57e71e09fbafbac3d140c89d9b3a3be786a33338429f945829a582cff0c91d83 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: 모든 학습 경로 → **평가·실패 진단·재현 감사**

$$\text{fair comparison}=\text{same init}+\text{same data}+\text{same budget}+\text{same metric contract}$$

좋은 score 하나는 reward hacking, length bias, entropy collapse를 숨길 수 있습니다. 공정 비교는 동일 initial hash, data order, token/forward/env budget, metric 정의를 요구합니다. local toy 결과와 논문 benchmark는 규모와 조건이 달라 같은 표에 순위처럼 놓지 않습니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** 세 report가 모두 성공 파일이어도 `result_origin`이 빠지면 local 실행 증거로 받아들일 수 있을까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>아니요. origin·환경·config·budget이 없으면 실행 결과와 예시 데이터를 구분할 수 없습니다.</details>

In [2]:
report_paths = [
    ROOT / "docs/research/C6_ALIGNMENT_BENCHMARK.json",
    ROOT / "docs/research/C7_GROUP_BENCHMARK.json",
    ROOT / "docs/research/C9_AGENTIC_BENCHMARK.json",
]
audit_rows = []
for report_path in report_paths:
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    audit_rows.append({
        "report": report_path.name,
        "origin": payload.get("result_origin"),
        "has_sources": bool(payload.get("sources")),
        "has_guardrail": bool(payload.get("interpretation") or payload.get("interpretation_guardrails")),
    })
print(audit_rows)

[{'report': 'C6_ALIGNMENT_BENCHMARK.json', 'origin': 'local_executed', 'has_sources': False, 'has_guardrail': True}, {'report': 'C7_GROUP_BENCHMARK.json', 'origin': 'local_executed', 'has_sources': True, 'has_guardrail': True}, {'report': 'C9_AGENTIC_BENCHMARK.json', 'origin': 'local_executed', 'has_sources': True, 'has_guardrail': True}]


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** capstone은 새 대규모 train보다 기존 artifact의 계약을 기계적으로 읽습니다. 더 많은 seed는 통계 불확실성을 줄이지만 비교 조건 불일치를 고치지는 못합니다.

**흔한 함정:** 서로 다른 token budget을 같은 `steps`로 비교하면 긴 response 알고리즘에 계산량 이점이 생깁니다. 여러 budget counter와 split hash를 같이 보고합니다. 회귀 test: `test_alignment_comparison_audits_shared_start_and_prompts`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert len(audit_rows) == 3
assert all(row["origin"] == "local_executed" for row in audit_rows)
print("checks=passed")

checks=passed


**회상 문제:** 한 알고리즘의 exact-match가 높고 entropy가 0에 가까우면 어떤 두 해석을 추가로 구분해야 하나요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** C6·C7·C9 report 모두 `local_executed`와 해석 guardrail을 가졌습니다. C6은 report 내부 `sources` 배열이 없어 source traceability 보강이 필요하다는 gap도 드러났습니다.
- 실제 확인: `test_alignment_comparison_audits_shared_start_and_prompts`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. 이제 README의 빠른/전체 경로로 돌아가 약한 영역을 복습하고, reproducibility checklist로 자신의 실험을 설계합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`
- `ppo-2017` — `docs/sources.yml`
- `dpo-2023` — `docs/sources.yml`
- `deepseekmath-grpo-2024` — `docs/sources.yml`
- `dapo-2025` — `docs/sources.yml`
- `agent-lightning-2025` — `docs/sources.yml`